<h2>Impact (v0.5) - 'Impact (v0.5).ipynb'</h2>

The aim of the 'Impact' notebooks are to address the following questions laid out in Q1.1.4 of 'Working Paper 1' focusing on 'Task 1.1 - GFI Trajectories and Relevance to African Shipping'.

<u><h4>How do the fleet-level transport cost outputs generated in '3. LEAP III Inventories (v0.1).ipynb' map to commodity flow volumes and GDP-level welfare effects?</h4></u>

We will first focus on the mapping to commodity flows, for which we will need seaborne trade portfolios as a first step. Once we have these, we can then proceed with the mapping step between fleet-level transport cost outputs and commodity flows. To achieve this, we will first integrate Sea Distances derived in the 'CERDI Sea Distances' table. We can then derive the product of the sea distance with the cargo volume, and multiply this by the transport cost intensities identified in the matching '3. LEAP III Inventories (v0.1).ipynb' notebook.

<u><h4> How sensitive are these estimates to data availability for African trade routes where vessel-tracking data is incomplete?</h4></u>

The seaborne trade portfolios are here derived from UNCTAD's Trade-and-Transport database. This primarily owes to the fact that the database is able to provide bilateral seaborne trade portfolios 'off-the-shelf' to users, however there are limitations with the data within the dataset that principally owe to the quality of input Comtrade data utilised for the year in question. Care should therefore taken to explore and understand the quality associated with the Trade-and-Transport-derived seaborne trade portfolio derived.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios.

Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.5' is amended to work with Comtrade import and export trade portfolios downloaded from the platform.

Limitations of the method include the following:

<ol type=1>
    <li>There are inherent limitations when using the Trade-and-Transport dataset, further details of which can be found its underlying documentation.</li>
    <li>2018 data is used, which is increasingly out of date and may not reflect the latest changes impacting each of the country's of interest.</li>
</ol>

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# Define functions for plotting a count of values in a given data field.
int_arr_voys_field_count = lambda field: int_arr_voys[field].value_counts().to_frame().reset_index()
int_dep_voys_field_count = lambda field: int_dep_voys[field].value_counts().to_frame().reset_index()

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.5"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"

# 1. Set up the notebook

## 1.1 Voyages with FEM Results

In [2]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv("/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_0.1.csv", dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 1.2 Let's set the country we're interested in and set up a results folder.

In [3]:
# Select one of our countries to work on...
iso_code = "404"
iso_country = "Kenya"
year = 2020
gdp = 100.660e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

'Output directory already exists.'

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_v0.6.csv".format(iso_country), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.netWgt.sum() / 1e6, 1)))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Kenya': 

Number of trade records present in dataset: 15466
Number of partner countries present in dataset: 151
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$12184.9m
Total weight within the dataset: 17738.3 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,1,404,Kenya,842,USA,000,World,Import,Sea,Carpets and other textile floor coverings; wov...,5702,m²,58.735,1216.291,944.164,NaN,944.164,Carpets and other te,5702 | Carpets and other te,USA | 5702 | Carpets and other te,57,57 | Carpets and other te,Carpets and other te,USA | 57 | Carpets and other te,[Container],18742.0,11645.735282,14164.603012
1,2,404,Kenya,724,Spain,000,World,Import,Sea,"Footwear; with outer soles of rubber, plastics...",6403,2u,8.888,12.970,266.330,NaN,266.330,Footwear; with outer,6403 | Footwear; with outer,Spain | 6403 | Footwear; with outer,64,64 | Footwear; gaiters an,Footwear; gaiters an,Spain | 64 | Footwear; gaiters an,[Container],9223.0,5730.904733,74.329834


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 1150
Total size of the fleet: 633 vessels
6 vessel types contained in the dataset: ['Bulk carrier' 'General cargo' 'Container' 'Vehicle' 'Chemical tanker'
 'Oil tanker']
Total transport work capacity: 131810.4 million deadweight-tonne nautical miles
Total cargo transported: 27069285.5 tonnes
Total transport work: 74852.8 million tonne nautical miles
Average build year of the fleet: 2006
Total Energy Demands: 11847.8 EJ
Total CO2 produced: 916965.6 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 2.3 Evaluate NZF Costs

In [14]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_arr_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_arr_voys_vess.shape[0] == 0:
                int_arr_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_arr_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
            
            if int_arr_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
                continue
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["lh_levy_usd_"+year] = lh_levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.netWgt.sum() / 1e6, 1)))


print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_im.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.lh_levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_na = trade_im[(trade_im[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im = trade_im[(~trade_im[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['8803 | Aircraft; parts of h']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

['Liquefied gas tanker']


Number of trade records present in dataset: 15466
Number of partner countries present in dataset: 151

Total value of trade: US$12184.9m
Total weight within the dataset: 17738.3 million kilograms


Total FEM costs in 2041...

BAU: US$533.3m (4.4% ad-val)
Levy: US$1133.8m (9.3% ad-val)
Levy (low-then-high): US$1106.3m (9.1% ad-val)
'as-is': US$1074.6m (8.8% ad-val)
Brazil: US$1107.2m (9.1% ad-val)
LISCR: US$742.1m (6.1% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,1,404,Kenya,842,USA,000,World,Import,Sea,Carpets and other textile floor coverings; wov...,5702,m²,58.735,1216.291,944.164,NaN,944.164,Carpets and other te,5702 | Carpets and other te,USA | 5702 | Carpets and other te,57,57 | Carpets and other te,Carpets and other te,USA | 57 | Carpets and other te,[Container],18742.0,11645.735282,14164.603012,96.227768,95.873300,95.886497,95.899724,95.899724,96.123917,101.110072,190.260165,116.956013,111.516105,102.925571,103.322563,98.792404,180.988021,129.819924,116.338224,107.054628,104.585478,96.761563,200.621349,190.947528,146.082824,127.182802,117.510097,94.759286,212.821257,195.403045,170.735238,159.297514,124.618359,92.793812,217.275002,209.491039,205.101181,208.571589,134.785103,90.790583,213.420117,212.992094,204.230311,216.859856,163.101889,88.904871,188.932178,199.638617,194.853285,203.865877,151.184038,86.988342,170.243703,171.572674,174.124430,201.315249,166.401755
1,2,404,Kenya,724,Spain,000,World,Import,Sea,"Footwear; with outer soles of rubber, plastics...",6403,2u,8.888,12.970,266.330,NaN,266.330,Footwear; with outer,6403 | Footwear; with outer,Spain | 6403 | Footwear; with outer,64,64 | Footwear; gaiters an,Footwear; gaiters an,Spain | 64 | Footwear; gaiters an,[Container],9223.0,5730.904733,74.329834,0.504963,0.503102,0.503172,0.503241,0.503241,0.504418,0.530583,0.998405,0.613736,0.585189,0.540110,0.542193,0.518421,0.949748,0.681240,0.610494,0.561777,0.548820,0.507764,1.052776,1.002012,0.766581,0.667401,0.616643,0.497257,1.116796,1.025392,0.895946,0.835926,0.653944,0.486943,1.140167,1.099320,1.076284,1.094495,0.707295,0.476431,1.119938,1.117692,1.071714,1.137989,0.855890,0.466535,0.991436,1.047619,1.022507,1.069802,0.793350,0.456478,0.893367,0.900341,0.913731,1.056417,0.873206


## 2.4 Inpsect Output

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,571.4,560.6,551.4,542.3,533.3,524.1,515.4,506.4
1,Levy,971.1,935.6,1031.4,1125.1,1133.8,1086.5,990.2,907.9
2,Levy (LH),640.2,702.4,1009.0,1061.2,1106.3,1098.6,1030.0,911.9
3,'as-is',612.9,640.2,772.2,898.7,1074.6,1058.2,1011.3,926.9
4,Brazil,574.4,597.9,698.3,847.2,1107.2,1119.1,1092.0,1059.8
5,LISCR,576.6,586.5,653.7,687.0,742.1,879.4,848.8,889.4



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,4.7,4.6,4.5,4.5,4.4,4.3,4.2,4.2
1,Levy,8.0,7.7,8.5,9.2,9.3,8.9,8.1,7.5
2,Levy (LH),5.3,5.8,8.3,8.7,9.1,9.0,8.5,7.5
3,'as-is',5.0,5.3,6.3,7.4,8.8,8.7,8.3,7.6
4,Brazil,4.7,4.9,5.7,7.0,9.1,9.2,9.0,8.7
5,LISCR,4.7,4.8,5.4,5.6,6.1,7.2,7.0,7.3



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.568,0.557,0.548,0.539,0.530,0.521,0.512,0.503
1,Levy,0.965,0.929,1.025,1.118,1.126,1.079,0.984,0.902
2,Levy (LH),0.636,0.698,1.002,1.054,1.099,1.091,1.023,0.906
3,'as-is',0.609,0.636,0.767,0.893,1.068,1.051,1.005,0.921
4,Brazil,0.571,0.594,0.694,0.842,1.100,1.112,1.085,1.053
5,LISCR,0.573,0.583,0.649,0.683,0.737,0.874,0.843,0.884


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,399.6,374.9,480.1,582.9,600.5,562.4,474.7,401.5
1,Levy (LH),68.8,141.8,457.6,518.9,572.9,574.5,514.6,405.5
2,'as-is',41.5,79.6,220.9,356.5,541.3,534.1,495.8,420.4
3,Brazil,3.0,37.2,147.0,305.0,573.9,595.0,576.6,553.4
4,LISCR,5.2,25.8,102.3,144.8,208.8,355.3,333.4,383.0



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,3.3,3.1,3.9,4.8,4.9,4.6,3.9,3.3
1,Levy (LH),0.6,1.2,3.8,4.3,4.7,4.7,4.2,3.3
2,'as-is',0.3,0.7,1.8,2.9,4.4,4.4,4.1,3.5
3,Brazil,0.0,0.3,1.2,2.5,4.7,4.9,4.7,4.5
4,LISCR,0.0,0.2,0.8,1.2,1.7,2.9,2.7,3.1



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.397,0.372,0.477,0.579,0.597,0.559,0.472,0.399
1,Levy (LH),0.068,0.141,0.455,0.516,0.569,0.571,0.511,0.403
2,'as-is',0.041,0.079,0.219,0.354,0.538,0.531,0.493,0.418
3,Brazil,0.003,0.037,0.146,0.303,0.570,0.591,0.573,0.550
4,LISCR,0.005,0.026,0.102,0.144,0.207,0.353,0.331,0.381


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_v0.6.csv".format(iso_country), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.netWgt.sum() / 1e6, 1)))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Kenya': 

Number of trade records present in dataset: 4013
Number of partner countries present in dataset: 143
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$2725.8m
Total weight within the dataset: 2144.4 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,404,Kenya,276,Germany,000,World,Export,Sea,"Thermionic, cold cathode or photo-cathode valv...",8540,u,0.0,1.0,NaN,1.202,1.202,"Thermionic, cold cat","8540 | Thermionic, cold cat","Germany | 8540 | Thermionic, cold cat",85,85 | Electrical machinery,Electrical machinery,Germany | 85 | Electrical machinery,[Container],12814.0,7962.247994,7.962248
1,1,404,Kenya,784,United Arab Emirates,000,World,Export,Sea,Machinery (not of heading no. 8450) for washin...,8451,u,0.0,10.0,NaN,0.564,0.564,Machinery (not of he,8451 | Machinery (not of he,United Arab Emirates | 8451 | Machinery (not o...,84,84 | Machinery and mechan,Machinery and mechan,United Arab Emirates | 84 | Machinery and mechan,"[Container, Bulk carrier]",4621.0,2871.355391,28.713554


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 1161
Total size of the fleet: 642 vessels
6 vessel types contained in the dataset: ['Bulk carrier' 'General cargo' 'Container' 'Chemical tanker' 'Vehicle'
 'Oil tanker']
Total transport work capacity: 91332.8 million deadweight-tonne nautical miles
Total cargo transported: 27432629.1 tonnes
Total transport work: 50685.6 million tonne nautical miles
Average build year of the fleet: 2006
Total Energy Demands: 7844.3 EJ
Total CO2 produced: 606985.5 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_dep_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_dep_voys_vess.shape[0] == 0:
                int_dep_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_dep_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

            if int_dep_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["lh_levy_usd_"+year] = lh_levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.netWgt.sum() / 1e6, 1)))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.lh_levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_na = trade_ex[(trade_ex[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(
    trade_ex_na.primaryValue.sum(), 
    round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)
))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['8803 | Aircraft; parts of h']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

['Liquefied gas tanker']


Number of trade records present in dataset: 4013
Number of partner countries present in dataset: 143

Total value of trade: US$2725.8m
Total weight within the dataset: 2144.4 million kilograms


Total FEM costs in 2041...

BAU: US$60.1m (2.2% ad-val)
Levy: US$129.3m (4.7% ad-val)
Levy (low-then-high): US$127.1m (4.7% ad-val)
'as-is': US$122.7m (4.5% ad-val)
Brazil: US$126.0m (4.6% ad-val)
LISCR: US$85.5m (3.1% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,404,Kenya,276,Germany,000,World,Export,Sea,"Thermionic, cold cathode or photo-cathode valv...",8540,u,0.0,1.0,NaN,1.202,1.202,"Thermionic, cold cat","8540 | Thermionic, cold cat","Germany | 8540 | Thermionic, cold cat",85,85 | Electrical machinery,Electrical machinery,Germany | 85 | Electrical machinery,[Container],12814.0,7962.247994,7.962248,0.054051,0.053851,0.053859,0.053866,0.053866,0.053993,0.056794,0.106900,0.065702,0.062649,0.057810,0.058033,0.055492,0.101675,0.072920,0.065348,0.060132,0.058746,0.054351,0.112702,0.10732,0.082059,0.071438,0.066006,0.053226,0.119559,0.109777,0.095901,0.089485,0.069995,0.052122,0.122053,0.117700,0.115191,0.117140,0.075715,0.050996,0.119892,0.119670,0.114747,0.121826,0.091598,0.049937,0.106144,0.112157,0.109451,0.114515,0.084947,0.048860,0.095631,0.096386,0.097820,0.113103,0.093474
1,1,404,Kenya,784,United Arab Emirates,000,World,Export,Sea,Machinery (not of heading no. 8450) for washin...,8451,u,0.0,10.0,NaN,0.564,0.564,Machinery (not of he,8451 | Machinery (not of he,United Arab Emirates | 8451 | Machinery (not o...,84,84 | Machinery and mechan,Machinery and mechan,United Arab Emirates | 84 | Machinery and mechan,"[Container, Bulk carrier]",4621.0,2871.355391,28.713554,0.181196,0.180666,0.180687,0.180709,0.180709,0.181027,0.189225,0.333915,0.214687,0.205049,0.191049,0.191734,0.185417,0.319436,0.235653,0.213018,0.198907,0.195071,0.182079,0.352458,0.34383,0.262940,0.233938,0.218101,0.178792,0.380285,0.359303,0.304231,0.285809,0.230092,0.175566,0.387442,0.379216,0.367464,0.375992,0.251648,0.172277,0.375639,0.377268,0.362061,0.383063,0.298448,0.169180,0.337393,0.355829,0.347598,0.367712,0.281734,0.166031,0.308745,0.310752,0.314986,0.361558,0.301696


## 3.4 Inspect Output

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,64.5,63.3,62.2,61.2,60.1,59.1,58.1,57.1
1,Levy,110.6,106.1,116.8,126.9,129.3,124.5,112.6,103.8
2,Levy (LH),72.6,79.3,115.2,121.4,127.1,125.5,118.6,104.4
3,'as-is',69.4,72.0,88.1,101.5,122.7,120.3,116.1,105.8
4,Brazil,64.9,67.6,79.1,95.6,126.0,127.3,123.3,121.1
5,LISCR,65.1,66.4,74.1,78.0,85.5,100.7,95.8,101.4



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.4,2.3,2.3,2.2,2.2,2.2,2.1,2.1
1,Levy,4.1,3.9,4.3,4.7,4.7,4.6,4.1,3.8
2,Levy (LH),2.7,2.9,4.2,4.5,4.7,4.6,4.4,3.8
3,'as-is',2.5,2.6,3.2,3.7,4.5,4.4,4.3,3.9
4,Brazil,2.4,2.5,2.9,3.5,4.6,4.7,4.5,4.4
5,LISCR,2.4,2.4,2.7,2.9,3.1,3.7,3.5,3.7



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.064,0.063,0.062,0.061,0.060,0.059,0.058,0.057
1,Levy,0.110,0.105,0.116,0.126,0.128,0.124,0.112,0.103
2,Levy (LH),0.072,0.079,0.114,0.121,0.126,0.125,0.118,0.104
3,'as-is',0.069,0.071,0.088,0.101,0.122,0.120,0.115,0.105
4,Brazil,0.064,0.067,0.079,0.095,0.125,0.126,0.123,0.120
5,LISCR,0.065,0.066,0.074,0.077,0.085,0.100,0.095,0.101


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,46.1,42.8,54.6,65.8,69.1,65.4,54.5,46.7
1,Levy (LH),8.1,16.0,52.9,60.2,67.0,66.4,60.5,47.4
2,'as-is',4.9,8.7,25.9,40.3,62.6,61.2,58.0,48.7
3,Brazil,0.4,4.3,16.9,34.5,65.9,68.2,65.3,64.0
4,LISCR,0.6,3.1,11.9,16.8,25.4,41.6,37.8,44.4



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.7,1.6,2.0,2.4,2.5,2.4,2.0,1.7
1,Levy (LH),0.3,0.6,1.9,2.2,2.5,2.4,2.2,1.7
2,'as-is',0.2,0.3,0.9,1.5,2.3,2.2,2.1,1.8
3,Brazil,0.0,0.2,0.6,1.3,2.4,2.5,2.4,2.3
4,LISCR,0.0,0.1,0.4,0.6,0.9,1.5,1.4,1.6



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.046,0.042,0.054,0.065,0.069,0.065,0.054,0.046
1,Levy (LH),0.008,0.016,0.053,0.060,0.067,0.066,0.060,0.047
2,'as-is',0.005,0.009,0.026,0.040,0.062,0.061,0.058,0.048
3,Brazil,0.000,0.004,0.017,0.034,0.065,0.068,0.065,0.064
4,LISCR,0.001,0.003,0.012,0.017,0.025,0.041,0.038,0.044
